In [2]:
import os
import glob
import pandas as pd

def inspect_swiss_roll(file_path):
    """
    Dynamically locates the table header in Agilent 34970A files, loads the CSV,
    and outputs shape, column structure, and baseline statistics.
    """
    header_row_index = 0
    
    # Locate the telemetry table header row
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if 'Scan Num' in line or '101 (' in line or 'Scan Swee' in line:
                header_row_index = idx
                break
                
    # Load dataset from the detected header row
    df = pd.read_csv(file_path, skiprows=header_row_index)
    
    # Clean column whitespace and drop completely empty rows or columns
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    return df

# ==========================================
# EXECUTION ON SWISS ROLL FOLDER
# ==========================================
swiss_path = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\Swiss Roll"
csv_files = glob.glob(os.path.join(swiss_path, "*.csv"))

print(f"Found {len(csv_files)} files in Swiss Roll folder. Running inspection...\n")

for i, file_path in enumerate(csv_files, 1):
    file_name = os.path.basename(file_path)
    
    try:
        df = inspect_swiss_roll(file_path)
        
        print(f"File {i}: {file_name}")
        print(f"   Shape: {df.shape[0]} rows by {df.shape[1]} columns")
        print(f"   Columns: {list(df.columns)}")
        print(f"   Missing Values: {df.isnull().sum().sum()} total nulls")
        
        # Select numeric columns for basic baseline stats
        numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
        if len(numeric_cols) > 0:
            print("\n   Baseline Data Summary (First 4 Numeric Columns):")
            print(df[numeric_cols[:4]].describe().loc[['mean', 'min', 'max', 'std']])
        
        print("\n" + "="*65 + "\n")
        
    except Exception as e:
        print(f"Could not read {file_name}: {e}\n")

Found 2 files in Swiss Roll folder. Running inspection...

File 1: 1781333575_2slpm 0.csv
   Shape: 276 rows by 15 columns
   Columns: ['Scan Sweep Time (Sec)', 'Scan Number', '101 (°C)', '102 (°C)', '103 (°C)', '104 (°C)', '105 (°C)', '106 (°C)', '107 (°C)', '108 (°C)', '109 (°C)', '110 (°C)', '111 (°C)', '112 (°C)', '113 (°C)']
   Missing Values: 0 total nulls

   Baseline Data Summary (First 4 Numeric Columns):
      Scan Number    101 (°C)    102 (°C)    103 (°C)
mean   138.500000  110.185399  410.711042  250.613614
min      1.000000   56.652981  114.249453  125.783432
max    276.000000  161.711410  724.226792  317.600720
std     79.818544   36.960390  186.547536   61.008511


File 2: 1781333976_2slpm 0.csv
   Shape: 276 rows by 15 columns
   Columns: ['Scan Sweep Time (Sec)', 'Scan Number', '101 (°C)', '102 (°C)', '103 (°C)', '104 (°C)', '105 (°C)', '106 (°C)', '107 (°C)', '108 (°C)', '109 (°C)', '110 (°C)', '111 (°C)', '112 (°C)', '113 (°C)']
   Missing Values: 0 total nulls

   

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================================================
# 1. ROBUST DATA LOADER & OVERLOAD SCRUBBER
# =========================================================
def load_swiss_roll_file(file_path):
    """
    Dynamically finds the telemetry table header in Agilent 34970A logs,
    loads the 13-channel data, and scrubs hardware overload errors (+9.9E+37).
    """
    header_idx = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if any(key in line for key in ['Scan Num', '101 (', 'Scan Swee']):
                header_idx = idx
                break
                
    df = pd.read_csv(file_path, skiprows=header_idx)
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    # Standardize timestamp column name
    for col in df.columns:
        if 'swee' in col.lower() or 'time' in col.lower():
            df.rename(columns={col: 'Timestamp'}, inplace=True)
            break
            
    # SCRUB OVERLOAD ERRORS: Convert Agilent error constants (+9.9E+37) to NaN
    sensor_cols = [col for col in df.columns if '°C' in col]
    for col in sensor_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].apply(lambda x: np.nan if (abs(x) > 10000 or x == np.inf) else x)
            
    return df, sensor_cols

# =========================================================
# 2. STATISTICAL ENGINE
# =========================================================
def compute_swiss_statistics(df, sensor_cols, file_name):
    """
    Computes univariate statistics for all 13 channels and outputs 
    the inter-channel correlation matrix.
    """
    sensor_df = df[sensor_cols]
    
    print(f"================================================================")
    print(f" FEATURE STATISTICAL SUMMARY: {file_name}")
    print(f"================================================================")
    
    stats_df = pd.DataFrame({
        'Mean': sensor_df.mean(),
        'Std Dev': sensor_df.std(),
        'Min': sensor_df.min(),
        '25% (Q1)': sensor_df.quantile(0.25),
        '50% (Median)': sensor_df.median(),
        '75% (Q3)': sensor_df.quantile(0.75),
        'Max': sensor_df.max(),
        'IQR': sensor_df.quantile(0.75) - sensor_df.quantile(0.25),
        'Skewness': sensor_df.skew(),
        'Kurtosis': sensor_df.kurtosis(),
        'Null Count (Overloads)': sensor_df.isnull().sum()
    })
    
    print(stats_df.round(2))
    
    print(f"\n--- Inter-Sensor Correlation Matrix (First 6 Channels) ---")
    corr_matrix = sensor_df.corr()
    print(corr_matrix.iloc[:6, :6].round(4))
    print("================================================================\n")
    
    return corr_matrix

# =========================================================
# 3. INTERACTIVE PLOTLY VISUALIZATIONS (PER CSV FILE)
# =========================================================
def plot_whole_dataset(df, sensor_cols, corr_matrix, file_name):
    """
    Renders the combined 13-channel time series and correlation heatmap for a single file.
    """
    x_axis = df['Timestamp'] if 'Timestamp' in df.columns else df.index
    
    # 1. Combined 13-Channel Time-Series Chart
    fig_time = px.line(
        df, 
        x=x_axis, 
        y=sensor_cols,
        title=f"Whole-Dataset Spatial Telemetry (All 13 Channels) — {file_name}",
        labels={'value': 'Temperature (°C)', 'variable': 'Sensor Channel', 'x': 'Scan Step / Time'}
    )
    fig_time.update_layout(
        hovermode='x unified',
        xaxis=dict(rangeslider=dict(visible=True), type='category')
    )
    fig_time.show()
    
    # 2. Inter-Sensor Correlation Heatmap (13x13 Grid)
    fig_corr = px.imshow(
        corr_matrix, 
        text_auto=".2f", 
        aspect="auto",
        color_continuous_scale='RdBu_r',
        title=f"13x13 Inter-Sensor Correlation Matrix — {file_name}",
        labels=dict(color="Pearson r")
    )
    fig_corr.show()

def plot_individual_features(df, sensor_cols, file_name):
    """
    Renders a 13-row subplot grid pairing each channel's time series 
    trend with its statistical box plot.
    """
    num_sensors = len(sensor_cols)
    fig = make_subplots(
        rows=num_sensors, 
        cols=2, 
        column_widths=[0.75, 0.25],
        subplot_titles=[item for sublist in [[f"{col} - Trend", f"{col} - Distribution"] for col in sensor_cols] for item in sublist],
        horizontal_spacing=0.08,
        vertical_spacing=0.04
    )
    
    colors = px.colors.qualitative.Alphabet
    x_axis = df['Timestamp'] if 'Timestamp' in df.columns else df.index
    
    for idx, col in enumerate(sensor_cols):
        row_num = idx + 1
        color = colors[idx % len(colors)]
        
        # Left Column: Individual Channel Time-Series Trend
        fig.add_trace(
            go.Scatter(
                x=x_axis, 
                y=df[col], 
                mode='lines', 
                name=col,
                line=dict(color=color, width=1.5),
                showlegend=False
            ),
            row=row_num, col=1
        )
        
        # Right Column: Statistical Box Plot (Outlier Bounds & Spread)
        fig.add_trace(
            go.Box(
                y=df[col].dropna(), 
                name=col, 
                marker_color=color,
                boxpoints='outliers',
                showlegend=False
            ),
            row=row_num, col=2
        )
        
        fig.update_yaxes(title_text="Temp (°C)", row=row_num, col=1)
        
    fig.update_layout(
        title_text=f"Individual Feature Inspection (All 13 Channels) — {file_name}",
        height=220 * num_sensors,  # Dynamically scales height for 13 rows
        width=1250,
        showlegend=False
    )
    fig.show()

# =========================================================
# 4. EXECUTION LOOP: PROCESS EACH CSV SEPARATELY
# =========================================================
swiss_dir = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\Swiss Roll"
csv_files = sorted(glob.glob(os.path.join(swiss_dir, "*.csv")))

print(f"Found {len(csv_files)} CSV files in Swiss Roll folder. Processing each separately...\n")

for i, sample_file in enumerate(csv_files, 1):
    file_name = os.path.basename(sample_file)
    print(f"\n======== PROCESSING FILE {i}/{len(csv_files)}: {file_name} ========")
    
    try:
        # 1. Load data and scrub overload errors
        df_swiss, sensors = load_swiss_roll_file(sample_file)
        
        # 2. Compute statistical metrics
        corr_matrix = compute_swiss_statistics(df_swiss, sensors, file_name)
        
        # 3. Generate Whole-Dataset Plots for this file
        plot_whole_dataset(df_swiss, sensors, corr_matrix, file_name)
        
        # 4. Generate Individual Feature Subplots for this file
        plot_individual_features(df_swiss, sensors, file_name)
        
    except Exception as e:
        print(f"❌ Error processing {file_name}: {e}")

Found 2 CSV files in Swiss Roll folder. Processing each separately...


======== PROCESSING FILE 1/2: 1781333575_2slpm 0.csv ========
 FEATURE STATISTICAL SUMMARY: 1781333575_2slpm 0.csv
            Mean  Std Dev     Min  25% (Q1)  50% (Median)  75% (Q3)     Max  \
101 (°C)  110.19    36.96   56.65     69.81        117.53    143.41  161.71   
102 (°C)  410.71   186.55  114.25    169.62        502.56    543.15  724.23   
103 (°C)  250.61    61.01  125.78    252.30        272.27    288.68  317.60   
104 (°C)  408.56   111.14  152.58    370.83        445.64    493.44  537.34   
105 (°C)     NaN      NaN     NaN       NaN           NaN       NaN     NaN   
106 (°C)   26.12     0.42   25.12     25.85         26.17     26.49   26.74   
107 (°C)  519.38    44.42  252.30    501.95        522.50    543.82  580.62   
108 (°C)  486.17    63.40  265.26    449.50        472.32    489.36  722.83   
109 (°C)  428.35    36.66  251.13    408.80        423.00    450.82  534.14   
110 (°C)  383.63    83.


======== PROCESSING FILE 2/2: 1781333976_2slpm 0.csv ========
 FEATURE STATISTICAL SUMMARY: 1781333976_2slpm 0.csv
            Mean  Std Dev     Min  25% (Q1)  50% (Median)  75% (Q3)     Max  \
101 (°C)  110.19    36.96   56.65     69.81        117.53    143.41  161.71   
102 (°C)  410.71   186.55  114.25    169.62        502.56    543.15  724.23   
103 (°C)  250.61    61.01  125.78    252.30        272.27    288.68  317.60   
104 (°C)  408.56   111.14  152.58    370.83        445.64    493.44  537.34   
105 (°C)     NaN      NaN     NaN       NaN           NaN       NaN     NaN   
106 (°C)   26.12     0.42   25.12     25.85         26.17     26.49   26.74   
107 (°C)  519.38    44.42  252.30    501.95        522.50    543.82  580.62   
108 (°C)  486.17    63.40  265.26    449.50        472.32    489.36  722.83   
109 (°C)  428.35    36.66  251.13    408.80        423.00    450.82  534.14   
110 (°C)  383.63    83.07  146.46    393.60        407.88    430.53  476.85   
111 (°C)  469.2